## 🧹 VACUUM y el Comportamiento de los Archivos Obsoletos

Hasta ahora hemos visto que cada operación realizada sobre una Delta Table genera nuevas versiones dentro del historial transaccional. Gracias a este mecanismo, Delta Lake puede ofrecer capacidades como:

* ✅ Time Travel
* ✅ Restore Table
* ✅ Auditoría de cambios
* ✅ Historial completo de operaciones

---

### 🤔 Pero... ¿qué ocurre con los archivos antiguos?

Recordemos que Delta Lake trabaja bajo un modelo de archivos inmutables. Esto significa que cuando ejecutamos operaciones como:

* INSERT
* UPDATE
* DELETE
* MERGE
* OPTIMIZE

los archivos existentes no son modificados directamente. En su lugar:

* 📦 Se generan nuevos archivos Parquet
* 📝 Se registra una nueva versión en el Transaction Log
* 🚫 Los archivos antiguos quedan marcados como obsoletos

Estos archivos continúan existiendo físicamente porque pueden ser necesarios para:

* ⏳ Time Travel
* 🔄 Restore Table
* 📜 Auditorías históricas

---

### 🚀 ¿Dónde entra VACUUM?

Con el paso del tiempo, una Delta Table puede acumular una gran cantidad de archivos obsoletos. Para liberar espacio de almacenamiento, Delta Lake proporciona el comando:

```sql id="jhsg0y"
VACUUM nombre_catalago.nombre_esquema.nombre_tabla;
```

VACUUM identifica los archivos que:

* ✅ Ya no son utilizados por la versión actual
* ✅ Han superado el período de retención configurado
* ✅ No son necesarios para reconstruir versiones recientes

Una vez identificados, dichos archivos son eliminados físicamente del almacenamiento.

---

### ⏳ Período de Retención

Por defecto, Delta Lake utiliza un período de retención de: `7 días`. Esto significa que, aunque ejecutemos VACUUM hoy, los archivos generados durante los últimos 7 días permanecerán protegidos.

¿Por qué?

Porque todavía podrían ser necesarios para:

* 📜 Consultas históricas
* ⏳ Time Travel
* 🔄 Restauraciones mediante RESTORE

---

### ⚠️ Una vez eliminados, no hay vuelta atrás

Cuando VACUUM elimina archivos obsoletos:

* ❌ Ya no podrán utilizarse para Time Travel
* ❌ Ya no podrán utilizarse para RESTORE
* ❌ No podrán recuperarse mediante Delta Lake

Por esta razón, VACUUM debe ejecutarse con criterio, especialmente en entornos donde el historial de datos tiene valor operativo o regulatorio.

### 🚀 Punto de Inicio en Databricks

Antes de trabajar con Delta Lake necesitamos una sesión de Spark activa.

Spark será el motor encargado de:

* ✅ Leer datos
* ✅ Transformarlos
* ✅ Procesarlos de forma distribuida
* ✅ Persistirlos como Delta Tables

In [0]:
from pyspark.sql import SparkSession # Puerta de entrada para trabajar con spark <-- SIEMPRE DEBEMOS IMPORTAR LA LLAVE MAESTRA QUE INICIA TODO.
from pyspark.sql.functions import *  # Funciones propias del módulo SQL de Spark, para trabajar sobre Dataframes.
spark = SparkSession.builder.appName("05Vacuum").getOrCreate() 
"""
^          ^__________^        ^_________^                               ^
|                |                   |                                   | 
Variable   Constructor de Sesión   Nombre Aplicación       Evita conflicto del SparkSession"""

print("🚀 Spark Session iniciada correctamente")

#### 🔍 ¿Qué acaba de ocurrir?

Acabamos de crear una Spark Session. Esta sesión representa nuestro punto de entrada hacia:
* 🏗️ Apache Spark
* 🏗️ Databricks Runtime
* 🏗️ Delta Lake
* 🏗️ Unity Catalog

### ⚡ Utilizaremos la Delta Table DIFERENTE

- Nombre: `bds_relacionales.northwind.region`

> En este caso, la tabla que utilizaré la definí en el año 2025. Esto me permitirá ejemplificar mejor `VACUUM`

In [0]:
## VERIFICAR HISTORIAL DE LA TABLA
display(spark.sql("DESCRIBE HISTORY bds_relacionales.northwind.region")) ## Resultado ⬇️

#### ⬇️ Resultado: Verificamos el historial hasta el 2025.
|version|timestamp|userId|userName|operation|operationParameters|job|notebook|queryHistoryStatementId|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics|userMetadata|engineInfo|
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
|1|2025-08-27T04:12:23.000+00:00|4105283318563748|bryanneciosup626@gmail.com|WRITE|{"mode":"Append","statsOnLoad":"true","partitionBy":"[]"}|null|null|null|null|0|WriteSerializable|true|{"numFiles":"1","numOutputRows":"5","numOutputBytes":"1008"}|null|Databricks-Runtime/17.0.x-aarch64-photon-scala2.13|
|0|2025-08-27T04:11:57.000+00:00|4105283318563748|bryanneciosup626@gmail.com|CREATE TABLE|{"partitionBy":"[]","clusterBy":"[]","description":null,"isManaged":"true","properties":"{\"delta.checkpoint.writeStatsAsJson\":\"false\",\"delta.checkpoint.writeStatsAsStruct\":\"true\",\"delta.enableDeletionVectors\":\"true\"}","statsOnLoad":"false"}|null|null|null|null|null|WriteSerializable|true|{}|null|Databricks-Runtime/17.0.x-aarch64-photon-scala2.13|

In [0]:
## REGISTRAMOS NUEVA INFORMACIÓN PARA ESTE AÑO 2026
spark.sql("INSERT INTO bds_relacionales.northwind.region(region_id,region_description)VALUES(6,'Latam-Sur');")
spark.sql("INSERT INTO bds_relacionales.northwind.region(region_id,region_description)VALUES(7,'Latam-Centro');")
print("Datos registrados correctamente")

## VERIFICAR HISTORIAL DE LA TABLA
display(spark.sql("DESCRIBE HISTORY bds_relacionales.northwind.region")) ## Resultado ⬇️

#### ⬇️ Resultado: Verificamos el historial (2025-2026)
|version|timestamp|userId|userName|operation|operationParameters|job|notebook|queryHistoryStatementId|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics|userMetadata|engineInfo|
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
|3|2026-06-18T22:25:33.000+00:00|4105283318563748|bryanneciosup626@gmail.com|WRITE|{"mode":"Append","statsOnLoad":"true","partitionBy":"[]"}|null|{"notebookId":"2191576506036385"}|9e3f259a-31d2-4299-a06f-becb4e2f8f31|0618-220221-s311jlri-v2n|2|WriteSerializable|true|{"numFiles":"1","numOutputRows":"1","numOutputBytes":"1143"}|null|Databricks-Runtime/18.2.x-aarch64-photon-scala2.13|
|2|2026-06-18T22:25:31.000+00:00|4105283318563748|bryanneciosup626@gmail.com|WRITE|{"mode":"Append","statsOnLoad":"true","partitionBy":"[]"}|null|{"notebookId":"2191576506036385"}|28ebba96-8e83-4b7a-9029-1a190e46ddd7|0618-220221-s311jlri-v2n|1|WriteSerializable|true|{"numFiles":"1","numOutputRows":"1","numOutputBytes":"1127"}|null|Databricks-Runtime/18.2.x-aarch64-photon-scala2.13|
|1|2025-08-27T04:12:23.000+00:00|4105283318563748|bryanneciosup626@gmail.com|WRITE|{"mode":"Append","statsOnLoad":"true","partitionBy":"[]"}|null|null|null|null|0|WriteSerializable|true|{"numFiles":"1","numOutputRows":"5","numOutputBytes":"1008"}|null|Databricks-Runtime/17.0.x-aarch64-photon-scala2.13|
|0|2025-08-27T04:11:57.000+00:00|4105283318563748|bryanneciosup626@gmail.com|CREATE TABLE|{"partitionBy":"[]","clusterBy":"[]","description":null,"isManaged":"true","properties":"{\"delta.checkpoint.writeStatsAsJson\":\"false\",\"delta.checkpoint.writeStatsAsStruct\":\"true\",\"delta.enableDeletionVectors\":\"true\"}","statsOnLoad":"false"}|null|null|null|null|null|WriteSerializable|true|{}|null|Databricks-Runtime/17.0.x-aarch64-photon-scala2.13|

In [0]:
## APLICAR VACUUM
spark.sql("VACUUM bds_relacionales.northwind.region")
print("VACUUM ejecutado correctamente")
### 💡 En este caso, VACUUM solo podrá proteger los archivos de 7 días hacia atrás.

## REALIZAR TIME TRAVEL POR VERSIÓN (VERSIÓN 1 - 2025)
display(spark.sql("SELECT * FROM bds_relacionales.northwind.region VERSION AS OF 1"))
"""
Resultado: [DELTA_UNSUPPORTED_TIME_TRAVEL_BEYOND_DELETED_FILE_RETENTION_DURATION] Cannot time travel beyond delta.deletedFileRetentionDuration (168 HOURS) set on the table. SQLSTATE: 0AKDC.

Este resultado indica que:

* VACUUM no elimina las versiones registradas en el Transaction Log.
* VACUUM elimina físicamente los archivos de datos obsoletos que permiten reconstruir esas versiones.
* Por ello, después de ejecutar VACUUM, es posible seguir viendo versiones antiguas en DESCRIBE HISTORY, pero no realizar TIME TRAVEL hacia ellas si los archivos necesarios ya fueron eliminados según la política de retención configurada.
"""
print()


In [0]:
## MODIFICAR EL PERIODO DE RETENCIÓN DEL VACUUM

"""
💡 EN CASO NECESITEMOS MODIFICAR EL PERIODO DE RETENCIÓN DEL VACUUM
    DE 7 DÍAS A MENOS. SE PUEDE MODIFICAR MEDIANTE:

    1. Deshabilitar verificación de retención:
    spark.databricks.delta.retentionDurationCheck.enabled = false;
    
    2. Personalizar el periodo de retención:
    VACUUM nombre_catalago.nombre_esquema.nombre_tabla RETAIN N HOURS;

        * N: ES LA CANTIDAD DE HORAS A MODIFICAR EL PERIODO DE RETENCIÓN.

"""
print()

### 🎓 Notas finales

Mientras que el Transaction Log mantiene el historial lógico de la tabla, VACUUM se encarga de limpiar los archivos físicos que ya no son necesarios.

En otras palabras:

* 📝 Delta Lake conserva el historial
* 🧹 VACUUM libera almacenamiento
* ⚖️ El período de retención define el equilibrio entre capacidad de recuperación y uso eficiente del espacio